In [1]:
import sys
print(sys.executable)
print(sys.version)

C:\Users\Bruno\anaconda3\python.exe
3.9.13 (main, Aug 25 2022, 23:51:50) [MSC v.1916 64 bit (AMD64)]


In [4]:
!python -m pip uninstall -y pyspark py4j

!python -m pip install install-jdk
!python -m pip install pyspark==3.3.2 py4j==0.10.9.5

!python -c "import jdk, pathlib; path = pathlib.Path.home() / '.jdks' / 'jdk8'; path.mkdir(parents=True, exist_ok=True); installed_path = jdk.install('8', path=str(path)); print('JAVA8_INSTALLED_PATH=' + installed_path)"

Found existing installation: pyspark 3.3.2
Uninstalling pyspark-3.3.2:
  Successfully uninstalled pyspark-3.3.2
Found existing installation: py4j 0.10.9.5
Uninstalling py4j-0.10.9.5:
  Successfully uninstalled py4j-0.10.9.5
  Using cached pyspark-3.3.2-py2.py3-none-any.whl
  Using cached py4j-0.10.9.5-py2.py3-none-any.whl (199 kB)
JAVA8_INSTALLED_PATH=C:\Users\Bruno\.jdks\jdk8\jdk8u492-b09


In [ ]:
table_paths = {
    "NOME_TABELA_1": "/caminho/tabela_1",
    "NOME_TABELA_2": "/caminho/tabela_2",
    "NOME_TABELA_3": "/caminho/tabela_3",
}

In [ ]:
table_paths = {
    "NOME_TABELA_1": "/caminho/tabela_1",
    "NOME_TABELA_2": "/caminho/tabela_2",
    "NOME_TABELA_3": "/caminho/tabela_3",
}

In [ ]:
specs_config = {
    "NOME_TABELA_1": {
        "pk_cols": ["COLUNA_PK"],
        "static": True,
    },

    "NOME_TABELA_2": {
        "pk_cols": ["COLUNA_PK"],
        "foreign_keys": [
            {
                "columns": ["COLUNA_FK"],
                "parent_table": "NOME_TABELA_1",
                "parent_columns": ["COLUNA_PK"],
            }
        ],
    },

    "NOME_TABELA_3": {
        "pk_cols": ["COLUNA_PK"],
        "foreign_keys": [
            {
                "columns": ["COLUNA_FK_1", "COLUNA_FK_2"],
                "parent_table": "NOME_TABELA_2",
                "parent_columns": ["COLUNA_PK_1", "COLUNA_PK_2"],
            }
        ],
    },
}

In [ ]:
synthetic = run_synthesis_from_paths(
    spark=spark,
    table_paths=table_paths,
    specs_config=specs_config,
    default_input_format="parquet",
    seed=42,
    validate_mode="full",
    save_path="/caminho/saida",
    save_format="parquet",
    verbose=True,
)

In [9]:
import os
import sys
import subprocess

JAVA_HOME = r"C:\Users\Bruno\.jdks\jdk8\jdk8u492-b09"

os.environ["JAVA_HOME"] = JAVA_HOME
os.environ["PATH"] = os.path.join(JAVA_HOME, "bin") + os.pathsep + os.environ["PATH"]

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print("JAVA_HOME:", os.environ["JAVA_HOME"])
print("Python:", sys.executable)

subprocess.run(["java", "-version"])

JAVA_HOME: C:\Users\Bruno\.jdks\jdk8\jdk8u492-b09
Python: C:\Users\Bruno\anaconda3\python.exe


CompletedProcess(args=['java', '-version'], returncode=0)

In [10]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("teste-spark-local")
    .master("local[1]")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.default.parallelism", "2")
    .config("spark.driver.memory", "1g")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.host", "127.0.0.1")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark OK:", spark.version)

Spark OK: 3.3.2


In [ ]:
# Importa o módulo
from gerador_cdb import generate_cdb_tables, validate_relationships

tipo_if, instrumento, operacao = generate_cdb_tables(
    spark,
    n_instrumentos=1000,
    n_operacoes=50000,
    seed=42,
)

tipo_if.show()
instrumento.show(5)
operacao.show(5)

# Valida
validate_relationships(tipo_if, instrumento, operacao)

Gerando TIPO_IF (5 linhas)...
Gerando INSTRUMENTO_FINANCEIRO (1000 linhas)...
Gerando OPERACAO (50000 linhas)...
+-----------+-----------+--------------------+-------------------+-------------------+
|NUM_TIPO_IF|COD_TIPO_IF|         NOM_TIPO_IF|       DAT_INCLUSAO|      DAT_ALTERACAO|
+-----------+-----------+--------------------+-------------------+-------------------+
|          1|        CDB|Certificado de De...|2019-12-01 00:00:00|2019-12-01 00:00:00|
|          2|        LCI|Letra de Crédito ...|2019-12-01 00:00:00|2019-12-01 00:00:00|
|          3|        LCA|Letra de Crédito ...|2019-12-01 00:00:00|2019-12-01 00:00:00|
|          4|         LF|    Letra Financeira|2019-12-01 00:00:00|2019-12-01 00:00:00|
|          5|       DPGE|Depósito a Prazo ...|2019-12-01 00:00:00|2019-12-01 00:00:00|
+-----------+-----------+--------------------+-------------------+-------------------+

+------+---------+------------+-----------+-------------------+-----------------+-------------------+--

### Apply

In [13]:
tipo_if.show(1)

+-----------+-----------+--------------------+-------------------+-------------------+
|NUM_TIPO_IF|COD_TIPO_IF|         NOM_TIPO_IF|       DAT_INCLUSAO|      DAT_ALTERACAO|
+-----------+-----------+--------------------+-------------------+-------------------+
|          1|        CDB|Certificado de De...|2019-12-01 00:00:00|2019-12-01 00:00:00|
+-----------+-----------+--------------------+-------------------+-------------------+
only showing top 1 row



In [14]:
instrumento.show(1)

+------+---------+------------+-----------+-------------------+-----------------+-------------------+-------------------+
|NUM_IF|   COD_IF|    COD_ISIN|NUM_TIPO_IF|VAL_NOMINAL_EMISSAO|VAL_NOMINAL_ATUAL|       DAT_INCLUSAO|      DAT_ALTERACAO|
+------+---------+------------+-----------+-------------------+-----------------+-------------------+-------------------+
|     1|LCI000001|BR6B86B273FF|          2|          219560.43|        272158.34|2024-10-12 12:17:27|2024-10-27 03:37:09|
+------+---------+------------+-----------+-------------------+-----------------+-------------------+-------------------+
only showing top 1 row



In [15]:
operacao.show(1)

+---------------+------+---------+-------------------+------------+------------------+--------------+---------------------+-------------------+
|NUM_ID_OPERACAO|NUM_IF|   COD_IF|       DAT_OPERACAO|QTD_OPERACAO|VAL_PRECO_UNITARIO|VAL_FINANCEIRO|COD_SITUACAO_OPERACAO|       DAT_INCLUSAO|
+---------------+------+---------+-------------------+------------+------------------+--------------+---------------------+-------------------+
|              1|   384|CDB000384|2022-03-22 06:11:42|         394|         9680.1299|    3813971.18|            LIQUIDADA|2022-06-16 17:06:03|
+---------------+------+---------+-------------------+------------+------------------+--------------+---------------------+-------------------+
only showing top 1 row



In [20]:
"""
synthetic_multitable_spark_v2.py
================================

Gerador de dados sintéticos MULTI-TABELA em PySpark.

Objetivo:
    Gerar dados sintéticos preservando:
      - estrutura relacional;
      - primary keys únicas;
      - foreign keys válidas;
      - distribuições por bootstrap de linhas inteiras;
      - relacionamentos entre tabelas via remapeamento old_key -> synthetic_key.

Características:
    1. PySpark.
    2. Suporta múltiplas tabelas.
    3. Suporta FK simples e composta.
    4. Suporta DAG de dependências entre tabelas.
    5. Suporta tabela estática/dimensão.
    6. Permite postprocess por tabela.
    7. Valida PK/FK com modos configuráveis.
    8. Evita broadcast obrigatório de mappings grandes.
    9. Usa cache com MEMORY_AND_DISK.
   10. Falha de forma explícita para tipos de PK inseguros.

Limitações:
    - Self-reference não é suportado.
    - Ciclos relacionais não são suportados.
    - Bootstrap de linhas inteiras pode repetir linhas.
    - Não é anonimização forte por si só.
    - Para 100+ tabelas grandes, recomenda-se reduzir validação full.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from functools import reduce
from typing import Callable, Dict, List, Mapping, Optional, Tuple, Literal
import warnings
import zlib

from pyspark import StorageLevel
from pyspark.sql import DataFrame, SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql import types as T


# ============================================================
# 1. Tipagens e specs
# ============================================================

NullableFkPolicy = Literal["allow_any_null", "allow_all_null", "invalid_null"]
ValidateMode = Literal["none", "full"]


@dataclass(frozen=True)
class ForeignKeySpec:
    """
    Declara uma FK do filho apontando para um pai.

    Exemplo de FK simples:
        ForeignKeySpec(
            columns=("NUM_TIPO_IF",),
            parent_table="tipo_if",
            parent_columns=("NUM_TIPO_IF",),
        )

    Exemplo de FK composta:
        ForeignKeySpec(
            columns=("NUM_IF", "COD_IF"),
            parent_table="instrumento",
            parent_columns=("NUM_IF", "COD_IF"),
        )
    """
    columns: Tuple[str, ...]
    parent_table: str
    parent_columns: Tuple[str, ...]


PostProcessor = Callable[[DataFrame, Mapping[str, DataFrame]], DataFrame]


@dataclass(frozen=True)
class TableSpec:
    """
    Especificação declarativa de uma tabela.

    Args:
        name:
            Nome lógico da tabela.
        pk_cols:
            Colunas que formam a primary key.
        foreign_keys:
            FKs da tabela.
        static:
            Se True, a tabela é copiada como está.
        postprocess:
            Função opcional aplicada depois de PK e FK serem ajustadas.
            Assinatura:
                postprocess(work_df, generated_tables) -> work_df
    """
    name: str
    pk_cols: Tuple[str, ...]
    foreign_keys: Tuple[ForeignKeySpec, ...] = field(default_factory=tuple)
    static: bool = False
    postprocess: Optional[PostProcessor] = None


# ============================================================
# 2. Utilitários de tipo e seed
# ============================================================

def _stable_seed(base_seed: int, *parts: object) -> int:
    """
    Gera seed determinístico sem depender de hash() do Python.
    """
    txt = "|".join(str(p) for p in (base_seed,) + parts)
    return int(zlib.crc32(txt.encode("utf-8")) % 2_000_000_000)


def _is_integer_type(dt: T.DataType) -> bool:
    return isinstance(dt, (T.ByteType, T.ShortType, T.IntegerType, T.LongType))


def _is_string_type(dt: T.DataType) -> bool:
    return isinstance(dt, T.StringType)


def _is_safe_pk_type(dt: T.DataType) -> bool:
    return _is_integer_type(dt) or _is_string_type(dt)


def _get_field_type(df: DataFrame, col_name: str) -> T.DataType:
    for f in df.schema.fields:
        if f.name == col_name:
            return f.dataType
    raise ValueError(f"Coluna `{col_name}` não existe no DataFrame.")


def _persist(df: DataFrame, storage_level: StorageLevel) -> DataFrame:
    return df.persist(storage_level)


def _safe_unpersist(df: Optional[DataFrame]) -> None:
    if df is None:
        return
    try:
        df.unpersist()
    except Exception:
        pass


# ============================================================
# 3. Validação das specs
# ============================================================

def _validate_specs(
    tables: Mapping[str, DataFrame],
    specs: Mapping[str, TableSpec],
) -> None:
    """
    Valida estrutura das specs antes de executar Spark jobs pesados.
    """
    if not specs:
        raise ValueError("`specs` está vazio.")

    for name, spec in specs.items():
        if name not in tables:
            raise ValueError(f"Tabela `{name}` está em specs, mas não está em tables.")

        if spec.name != name:
            raise ValueError(
                f"Inconsistência: chave specs=`{name}`, mas TableSpec.name=`{spec.name}`."
            )

        if not spec.pk_cols:
            raise ValueError(f"Tabela `{name}` precisa ter pelo menos uma coluna de PK.")

        df_cols = set(tables[name].columns)

        for pk in spec.pk_cols:
            if pk not in df_cols:
                raise ValueError(
                    f"PK col `{pk}` não existe na tabela `{name}`. "
                    f"Colunas disponíveis: {sorted(df_cols)}"
                )

        seen_fk_cols: set = set()

        for fk in spec.foreign_keys:
            if not fk.columns:
                raise ValueError(f"FK vazia declarada na tabela `{name}`.")

            if len(fk.columns) != len(fk.parent_columns):
                raise ValueError(
                    f"FK inválida em `{name}`: columns={fk.columns}, "
                    f"parent_columns={fk.parent_columns}. Tamanhos diferentes."
                )

            if fk.parent_table == name:
                raise ValueError(
                    f"Self-reference não suportado: tabela `{name}` referencia ela mesma."
                )

            if fk.parent_table not in specs:
                raise ValueError(
                    f"FK em `{name}` referencia `{fk.parent_table}`, "
                    f"mas essa tabela não existe em specs."
                )

            if fk.parent_table not in tables:
                raise ValueError(
                    f"FK em `{name}` referencia `{fk.parent_table}`, "
                    f"mas essa tabela não existe em tables."
                )

            for c in fk.columns:
                if c not in df_cols:
                    raise ValueError(
                        f"FK col `{c}` não existe na tabela filha `{name}`."
                    )

                if c in seen_fk_cols:
                    raise ValueError(
                        f"Coluna `{c}` em `{name}` participa de mais de uma FK. "
                        f"Este framework evita remapeamento ambíguo. "
                        f"Se isso for necessário, crie postprocess customizado."
                    )

                seen_fk_cols.add(c)

            parent_cols = set(tables[fk.parent_table].columns)
            for pc in fk.parent_columns:
                if pc not in parent_cols:
                    raise ValueError(
                        f"FK em `{name}` referencia coluna `{pc}`, "
                        f"mas ela não existe no pai `{fk.parent_table}`."
                    )


def _topological_order(specs: Mapping[str, TableSpec]) -> List[str]:
    """
    Ordena as tabelas garantindo que pais sejam gerados antes dos filhos.
    """
    remaining = set(specs.keys())
    done: set = set()
    order: List[str] = []

    while remaining:
        ready = [
            name
            for name in remaining
            if {fk.parent_table for fk in specs[name].foreign_keys}.issubset(done)
        ]

        if not ready:
            unresolved = {
                table: [fk.parent_table for fk in specs[table].foreign_keys]
                for table in remaining
            }
            raise ValueError(
                "Ciclo relacional, self-reference ou pai ausente detectado. "
                f"Pendências: {unresolved}"
            )

        for name in sorted(ready):
            order.append(name)
            done.add(name)
            remaining.remove(name)

    return order


def _referenced_parent_columns(
    specs: Mapping[str, TableSpec],
) -> Dict[str, set]:
    """
    Para cada tabela pai, lista quais conjuntos de colunas são referenciados.
    """
    refs: Dict[str, set] = {}

    for child_spec in specs.values():
        for fk in child_spec.foreign_keys:
            refs.setdefault(fk.parent_table, set()).add(tuple(fk.parent_columns))

    return refs


# ============================================================
# 4. Indexação e bootstrap
# ============================================================

def _with_contiguous_row_id(df: DataFrame, id_col: str) -> DataFrame:
    """
    Adiciona ID contíguo 0..N-1.

    Observação:
        Usa RDD zipWithIndex porque é mais adequado para índice contíguo
        do que Window global com row_number, que tende a concentrar em uma partição.

    Continua sendo PySpark, mas não é DataFrame SQL puro.
    """
    spark = df.sparkSession

    schema_with_id = T.StructType(
        df.schema.fields + [T.StructField(id_col, T.LongType(), False)]
    )

    indexed_rdd = df.rdd.zipWithIndex().map(
        lambda row_with_idx: tuple(row_with_idx[0]) + (row_with_idx[1],)
    )

    return spark.createDataFrame(indexed_rdd, schema=schema_with_id)


def _bootstrap_rows_exact(
    src_indexed: DataFrame,
    n_rows: int,
    *,
    src_count: int,
    seed: int,
    spark: SparkSession,
    keep_all_source_rows: bool,
) -> DataFrame:
    """
    Gera exatamente n_rows por bootstrap de linhas inteiras.

    Se keep_all_source_rows=True:
        Garante que toda linha original apareça pelo menos uma vez.
        Isso é necessário para tabela pai, pois assegura cobertura total
        do mapping old_key -> synthetic_key.

    Se keep_all_source_rows=False:
        Faz bootstrap uniforme puro.
    """
    if n_rows < 0:
        raise ValueError("n_rows deve ser >= 0.")

    src_cols = [c for c in src_indexed.columns if c != "__src_row_id"]

    if n_rows == 0:
        empty_schema = T.StructType(
            [
                T.StructField("__synthetic_pos", T.LongType(), False),
                T.StructField("__orig_src_row_id", T.LongType(), True),
            ]
            + [f for f in src_indexed.schema.fields if f.name in src_cols]
        )
        return spark.createDataFrame([], schema=empty_schema)

    if src_count == 0:
        raise ValueError(
            "A tabela fonte está vazia, mas n_rows > 0. "
            "Não há linhas para amostrar."
        )

    if keep_all_source_rows:
        if n_rows < src_count:
            raise ValueError(
                f"Tabela pai precisa de n_rows >= cardinalidade original. "
                f"Recebido n_rows={n_rows}, src_count={src_count}."
            )

        base_keep = (
            src_indexed
            .withColumn("__synthetic_pos", F.col("__src_row_id"))
            .withColumn("__orig_src_row_id", F.col("__src_row_id"))
            .select("__synthetic_pos", "__orig_src_row_id", *src_cols)
        )

        extra_n = n_rows - src_count

        if extra_n == 0:
            return base_keep

        extra_positions = (
            spark.range(src_count, n_rows)
            .withColumnRenamed("id", "__synthetic_pos")
            .withColumn(
                "__lookup_src_row_id",
                F.floor(F.rand(seed) * F.lit(src_count)).cast("long"),
            )
        )

        extra = (
            extra_positions
            .join(
                src_indexed,
                extra_positions["__lookup_src_row_id"] == src_indexed["__src_row_id"],
                "left",
            )
            .withColumn("__orig_src_row_id", F.col("__src_row_id"))
            .select("__synthetic_pos", "__orig_src_row_id", *src_cols)
        )

        return base_keep.unionByName(extra)

    positions = (
        spark.range(0, n_rows)
        .withColumnRenamed("id", "__synthetic_pos")
        .withColumn(
            "__lookup_src_row_id",
            F.floor(F.rand(seed) * F.lit(src_count)).cast("long"),
        )
    )

    return (
        positions
        .join(
            src_indexed,
            positions["__lookup_src_row_id"] == src_indexed["__src_row_id"],
            "left",
        )
        .withColumn("__orig_src_row_id", F.col("__src_row_id"))
        .select("__synthetic_pos", "__orig_src_row_id", *src_cols)
    )


# ============================================================
# 5. Geração de PK
# ============================================================

_INT_TYPE_LIMITS = (
    (T.ByteType, 127),
    (T.ShortType, 32_767),
    (T.IntegerType, 2_147_483_647),
)


def _max_pk_value(df_cached: DataFrame, pk: str) -> Optional[int]:
    row = df_cached.agg(F.max(F.col(pk)).alias("max_pk")).collect()[0]
    return int(row["max_pk"]) if row["max_pk"] is not None else None


def _set_unique_pk_column(
    work: DataFrame,
    source_cached: DataFrame,
    pk: str,
    *,
    append_after_max: bool,
    target_n: int,
    offset: int = 0,
) -> DataFrame:
    """
    Sobrescreve uma coluna PK com valores únicos.
    """
    dt = _get_field_type(source_cached, pk)

    if _is_integer_type(dt):
        start = (_max_pk_value(source_cached, pk) or 0) + 1 if append_after_max else 1
        highest = start + target_n - 1 + offset

        for type_cls, limit in _INT_TYPE_LIMITS:
            if isinstance(dt, type_cls) and highest > limit:
                raise OverflowError(
                    f"PK `{pk}` é {type_cls.__name__}, mas o maior valor sintético "
                    f"seria {highest:,}, excedendo o limite {limit:,}. "
                    f"Converta a coluna para LongType ou reduza n_rows."
                )

        return work.withColumn(
            pk,
            (F.col("__synthetic_pos") + F.lit(start + offset)).cast(dt),
        )

    if _is_string_type(dt):
        return work.withColumn(
            pk,
            F.concat(
                F.lit(f"SYN_{pk}_"),
                F.lpad(
                    (F.col("__synthetic_pos") + F.lit(offset)).cast("string"),
                    14,
                    "0",
                ),
            ).cast(dt),
        )

    raise TypeError(
        f"PK `{pk}` possui tipo {dt!r}, sem estratégia automática segura. "
        f"Use PK IntegerType, LongType ou StringType, ou implemente postprocess customizado."
    )


def _generate_pk_columns(
    work: DataFrame,
    source_cached: DataFrame,
    spec: TableSpec,
    *,
    append_after_max: bool,
    target_n: int,
) -> DataFrame:
    """
    Gera PK sintética.

    Estratégia:
        - PK simples: sobrescreve a única coluna.
        - PK composta:
            - se última coluna é int/string, usa última coluna como driver único;
            - caso contrário, lança erro.
    """
    if len(spec.pk_cols) == 1:
        return _set_unique_pk_column(
            work,
            source_cached,
            spec.pk_cols[0],
            append_after_max=append_after_max,
            target_n=target_n,
            offset=0,
        )

    last_pk = spec.pk_cols[-1]
    last_type = _get_field_type(source_cached, last_pk)

    if not _is_safe_pk_type(last_type):
        raise TypeError(
            f"PK composta da tabela `{spec.name}` usa última coluna `{last_pk}` "
            f"com tipo {last_type!r}. Para PK composta automática, a última coluna "
            f"precisa ser int ou string."
        )

    return _set_unique_pk_column(
        work,
        source_cached,
        last_pk,
        append_after_max=append_after_max,
        target_n=target_n,
        offset=0,
    )


# ============================================================
# 6. Mapping old -> new e remapeamento de FKs
# ============================================================

def _build_mapping_for_parent_cols(
    work_cached: DataFrame,
    parent_cols: Tuple[str, ...],
    storage_level: StorageLevel,
) -> DataFrame:
    """
    Cria mapping:

        chave original do pai -> chave sintética do pai

    Se uma chave original gerar vários candidatos sintéticos, cada candidato
    recebe rank. Depois o filho sorteia um rank válido.
    """
    old_cols = [f"__old__{c}" for c in parent_cols]

    missing_old = [c for c in old_cols if c not in work_cached.columns]
    if missing_old:
        raise ValueError(
            f"Não foi possível construir mapping. Colunas antigas ausentes: {missing_old}"
        )

    mapping = work_cached.select(
        *[
            F.col(old_cols[i]).alias(f"__old_{i}")
            for i in range(len(parent_cols))
        ],
        *[
            F.col(parent_cols[i]).alias(f"__new_{i}")
            for i in range(len(parent_cols))
        ],
        F.col("__synthetic_pos"),
    )

    partition_cols = [F.col(f"__old_{i}") for i in range(len(parent_cols))]

    w = Window.partitionBy(*partition_cols).orderBy(F.col("__synthetic_pos"))

    mapping = mapping.withColumn(
        "__candidate_rank",
        F.row_number().over(w).cast("long"),
    )

    counts = mapping.groupBy(
        *[F.col(f"__old_{i}") for i in range(len(parent_cols))]
    ).agg(
        F.count(F.lit(1)).cast("long").alias("__candidate_count")
    )

    mapping = mapping.join(
        counts,
        on=[f"__old_{i}" for i in range(len(parent_cols))],
        how="left",
    )

    return _persist(mapping, storage_level)


def _fk_join_condition(
    left_df: DataFrame,
    left_cols: List[str],
    right_df: DataFrame,
    right_cols: List[str],
):
    conditions = [
        left_df[left_cols[i]].eqNullSafe(right_df[right_cols[i]])
        for i in range(len(left_cols))
    ]
    return reduce(lambda a, b: a & b, conditions)


def _apply_fk_mapping(
    work: DataFrame,
    fk: ForeignKeySpec,
    mapping: DataFrame,
    *,
    seed: int,
    broadcast_fk_counts: bool,
) -> DataFrame:
    """
    Remapeia FK do filho para valores sintéticos existentes no pai.
    """
    fk_tag = (
        f"__fk_{fk.parent_table}_"
        f"{_stable_seed(seed, fk.parent_table, fk.columns, fk.parent_columns)}"
    )

    n = len(fk.columns)

    counts = mapping.select(
        *[
            F.col(f"__old_{i}").alias(f"{fk_tag}_old_{i}")
            for i in range(n)
        ],
        F.col("__candidate_count").alias(f"{fk_tag}_count"),
    ).dropDuplicates(
        [f"{fk_tag}_old_{i}" for i in range(n)]
    )

    count_old_cols = [f"{fk_tag}_old_{i}" for i in range(n)]
    cond_counts = _fk_join_condition(
        work,
        list(fk.columns),
        counts,
        count_old_cols,
    )

    if broadcast_fk_counts:
        work = work.join(F.broadcast(counts), cond_counts, "left")
    else:
        work = work.join(counts, cond_counts, "left")

    work = work.withColumn(
        f"{fk_tag}_rank",
        F.when(
            F.col(f"{fk_tag}_count").isNull(),
            F.lit(None).cast("long"),
        ).otherwise(
            F.floor(
                F.rand(_stable_seed(seed, fk_tag, "rank"))
                * F.col(f"{fk_tag}_count")
            ).cast("long") + F.lit(1)
        ),
    )

    m = mapping.select(
        *[
            F.col(f"__old_{i}").alias(f"{fk_tag}_map_old_{i}")
            for i in range(n)
        ],
        *[
            F.col(f"__new_{i}").alias(f"{fk_tag}_new_{i}")
            for i in range(n)
        ],
        F.col("__candidate_rank").alias(f"{fk_tag}_map_rank"),
    )

    map_old_cols = [f"{fk_tag}_map_old_{i}" for i in range(n)]

    cond_map_key = _fk_join_condition(
        work,
        list(fk.columns),
        m,
        map_old_cols,
    )

    cond_map = cond_map_key & (
        work[f"{fk_tag}_rank"] == m[f"{fk_tag}_map_rank"]
    )

    work = work.join(m, cond_map, "left")

    for i, child_col in enumerate(fk.columns):
        child_type = _get_field_type(work, child_col)
        work = work.withColumn(
            child_col,
            F.col(f"{fk_tag}_new_{i}").cast(child_type),
        )

    drop_cols = (
        [f"{fk_tag}_old_{i}" for i in range(n)]
        + [f"{fk_tag}_map_old_{i}" for i in range(n)]
        + [f"{fk_tag}_new_{i}" for i in range(n)]
        + [
            f"{fk_tag}_count",
            f"{fk_tag}_rank",
            f"{fk_tag}_map_rank",
        ]
    )

    return work.drop(*drop_cols)


# ============================================================
# 7. Validações de resultado
# ============================================================

def validate_primary_keys(
    tables: Mapping[str, DataFrame],
    specs: Mapping[str, TableSpec],
) -> DataFrame:
    """
    Retorna diagnóstico de PK por tabela.
    """
    spark = next(iter(tables.values())).sparkSession
    rows = []

    for name, spec in specs.items():
        df = tables[name]

        total_rows = df.count()
        distinct_pk = df.select(*spec.pk_cols).dropDuplicates().count()

        null_condition = reduce(
            lambda a, b: a | b,
            [F.col(c).isNull() for c in spec.pk_cols],
        )

        null_pk_rows = df.where(null_condition).count()
        duplicate_pk_rows = total_rows - distinct_pk

        rows.append(
            (
                name,
                ",".join(spec.pk_cols),
                int(total_rows),
                int(distinct_pk),
                int(null_pk_rows),
                int(duplicate_pk_rows),
            )
        )

    schema = (
        "table string, pk_cols string, total_rows long, distinct_pk long, "
        "null_pk_rows long, duplicate_pk_rows long"
    )

    return spark.createDataFrame(rows, schema=schema)


def _filter_child_fk_for_validation(
    child_df: DataFrame,
    fk: ForeignKeySpec,
    nullable_fk_policy: NullableFkPolicy,
) -> DataFrame:
    """
    Define como FKs nulas devem ser tratadas na validação.

    allow_any_null:
        Se qualquer coluna da FK for nula, ignora essa linha na validação.

    allow_all_null:
        Só ignora se todas as colunas da FK forem nulas.
        FK parcialmente nula será validada e tende a ser inválida.

    invalid_null:
        Não ignora nulos. Nulos podem aparecer como inválidos.
    """
    if nullable_fk_policy == "invalid_null":
        return child_df

    any_null = reduce(
        lambda a, b: a | b,
        [F.col(c).isNull() for c in fk.columns],
    )

    all_null = reduce(
        lambda a, b: a & b,
        [F.col(c).isNull() for c in fk.columns],
    )

    if nullable_fk_policy == "allow_any_null":
        return child_df.where(~any_null)

    if nullable_fk_policy == "allow_all_null":
        return child_df.where(~all_null)

    raise ValueError(f"nullable_fk_policy inválida: {nullable_fk_policy}")


def validate_foreign_keys(
    tables: Mapping[str, DataFrame],
    specs: Mapping[str, TableSpec],
    *,
    nullable_fk_policy: NullableFkPolicy = "allow_any_null",
) -> DataFrame:
    """
    Retorna diagnóstico de FK por relacionamento.
    """
    spark = next(iter(tables.values())).sparkSession
    rows = []

    for child_name, child_spec in specs.items():
        child_df_raw = tables[child_name]

        for fk in child_spec.foreign_keys:
            parent_df = tables[fk.parent_table]

            child_df = _filter_child_fk_for_validation(
                child_df_raw,
                fk,
                nullable_fk_policy,
            )

            child_keys = child_df.select(*fk.columns).dropDuplicates()

            parent_keys = parent_df.select(
                *[
                    F.col(parent_col).alias(child_col)
                    for child_col, parent_col in zip(fk.columns, fk.parent_columns)
                ]
            ).dropDuplicates()

            invalid = child_keys.join(
                parent_keys,
                on=list(fk.columns),
                how="left_anti",
            ).count()

            total_distinct = child_keys.count()

            rows.append(
                (
                    child_name,
                    ",".join(fk.columns),
                    fk.parent_table,
                    ",".join(fk.parent_columns),
                    int(total_distinct),
                    int(invalid),
                )
            )

    schema = (
        "child_table string, fk_cols string, parent_table string, parent_cols string, "
        "distinct_child_fk long, invalid_fk long"
    )

    return spark.createDataFrame(rows, schema=schema)


def _run_validation_or_raise(
    result: Mapping[str, DataFrame],
    specs: Mapping[str, TableSpec],
    *,
    nullable_fk_policy: NullableFkPolicy,
) -> None:
    pk_report = validate_primary_keys(result, specs)
    fk_report = validate_foreign_keys(
        result,
        specs,
        nullable_fk_policy=nullable_fk_policy,
    )

    pk_problems = pk_report.where(
        "null_pk_rows > 0 OR duplicate_pk_rows > 0"
    ).count()

    fk_problems = fk_report.where(
        "invalid_fk > 0"
    ).count()

    if pk_problems or fk_problems:
        print(">>> FALHA NA VALIDAÇÃO DE PK/FK")
        print(">>> PRIMARY KEYS")
        pk_report.show(truncate=False)

        print(">>> FOREIGN KEYS")
        fk_report.show(truncate=False)

        raise RuntimeError(
            f"Validação falhou: {pk_problems} tabela(s) com PK inválida, "
            f"{fk_problems} relação(ões) com FK inválida."
        )


# ============================================================
# 8. Função principal
# ============================================================

def synthesize_multitable_spark(
    tables: Mapping[str, DataFrame],
    specs: Mapping[str, TableSpec],
    n_rows_by_table: Optional[Mapping[str, int]] = None,
    *,
    seed: int = 42,
    append_after_max_pk: bool = True,
    validate_mode: ValidateMode = "full",
    nullable_fk_policy: NullableFkPolicy = "allow_any_null",
    broadcast_fk_counts: bool = False,
    storage_level: StorageLevel = StorageLevel.MEMORY_AND_DISK,
    verbose: bool = False,
) -> Dict[str, DataFrame]:
    """
    Gera dados sintéticos multi-tabela.

    Args:
        tables:
            Dict nome_tabela -> DataFrame original.
        specs:
            Dict nome_tabela -> TableSpec.
        n_rows_by_table:
            Dict nome_tabela -> número de linhas sintéticas.
            Se None, usa o volume original.
        seed:
            Semente global.
        append_after_max_pk:
            Se True, PK nova começa após max(PK original).
        validate_mode:
            "full" ou "none".
        nullable_fk_policy:
            Política para validar FKs nulas.
        broadcast_fk_counts:
            Se True, força broadcast da tabela de contagem de candidatos.
            Recomendado False para tabelas grandes.
        storage_level:
            Nível de persistência Spark.
        verbose:
            Se True, imprime progresso.

    Returns:
        Dict nome_tabela -> DataFrame sintético.
    """
    if validate_mode not in ("none", "full"):
        raise ValueError("validate_mode deve ser 'none' ou 'full'.")

    _validate_specs(tables, specs)

    n_rows_by_table = dict(n_rows_by_table or {})
    order = _topological_order(specs)
    parent_refs = _referenced_parent_columns(specs)

    result: Dict[str, DataFrame] = {}
    mappings: Dict[Tuple[str, Tuple[str, ...]], DataFrame] = {}
    intermediates: List[DataFrame] = []

    try:
        for table_name in order:
            source = tables[table_name]
            spec = specs[table_name]
            spark = source.sparkSession
            original_cols = source.columns

            target_n_raw = n_rows_by_table.get(table_name)

            ref_col_sets = parent_refs.get(table_name, set())

            ref_cols = sorted(
                set(c for cols in ref_col_sets for c in cols)
                | set(spec.pk_cols)
            )

            if spec.static:
                src_count = source.count()

                if target_n_raw is not None and int(target_n_raw) != src_count:
                    warnings.warn(
                        f"Tabela `{table_name}` é static=True. "
                        f"n_rows_by_table={target_n_raw} será ignorado. "
                        f"Serão copiadas {src_count} linhas."
                    )

                if verbose:
                    print(f"[{table_name}] STATIC | copiando {src_count} linhas")

                work = (
                    _with_contiguous_row_id(source, "__synthetic_pos")
                    .withColumn("__orig_src_row_id", F.col("__synthetic_pos"))
                )

                for c in ref_cols:
                    work = work.withColumn(f"__old__{c}", F.col(c))

                work = _persist(work, storage_level)
                work.count()
                intermediates.append(work)

            else:
                src_indexed = _with_contiguous_row_id(source, "__src_row_id")
                src_indexed = _persist(src_indexed, storage_level)
                src_count = src_indexed.count()
                intermediates.append(src_indexed)

                target_n = int(target_n_raw if target_n_raw is not None else src_count)
                keep_all = table_name in parent_refs

                if verbose:
                    role = "PAI" if keep_all else "FILHO"
                    print(
                        f"[{table_name}] {role} | origem={src_count:,} "
                        f"-> sintético={target_n:,}"
                    )

                work = _bootstrap_rows_exact(
                    src_indexed,
                    target_n,
                    src_count=src_count,
                    seed=_stable_seed(seed, table_name, "bootstrap"),
                    spark=spark,
                    keep_all_source_rows=keep_all,
                )

                for c in ref_cols:
                    work = work.withColumn(f"__old__{c}", F.col(c))

                work = _generate_pk_columns(
                    work,
                    src_indexed,
                    spec,
                    append_after_max=append_after_max_pk,
                    target_n=target_n,
                )

                for fk in spec.foreign_keys:
                    key = (fk.parent_table, tuple(fk.parent_columns))

                    if key not in mappings:
                        raise ValueError(
                            f"Mapping não encontrado para FK "
                            f"{table_name}.{fk.columns} -> "
                            f"{fk.parent_table}.{fk.parent_columns}"
                        )

                    work = _apply_fk_mapping(
                        work,
                        fk,
                        mappings[key],
                        seed=_stable_seed(
                            seed,
                            table_name,
                            fk.parent_table,
                            fk.columns,
                            fk.parent_columns,
                        ),
                        broadcast_fk_counts=broadcast_fk_counts,
                    )

                if spec.postprocess is not None:
                    work = spec.postprocess(work, result)

                work = _persist(work, storage_level)
                work.count()
                intermediates.append(work)

            if table_name in parent_refs:
                for cols in parent_refs[table_name]:
                    mapping_df = _build_mapping_for_parent_cols(
                        work,
                        tuple(cols),
                        storage_level=storage_level,
                    )
                    mapping_df.count()
                    mappings[(table_name, tuple(cols))] = mapping_df
                    intermediates.append(mapping_df)

            synth = work.select(*original_cols)
            synth = _persist(synth, storage_level)
            synth.count()

            result[table_name] = synth

        if validate_mode == "full":
            if verbose:
                print("Validando PKs e FKs...")

            _run_validation_or_raise(
                result,
                specs,
                nullable_fk_policy=nullable_fk_policy,
            )

            if verbose:
                print("Validação concluída: PKs e FKs OK.")

        return result

    except Exception:
        for df in result.values():
            _safe_unpersist(df)
        raise

    finally:
        for df in intermediates:
            _safe_unpersist(df)


# ============================================================
# 9. Postprocess específico das suas tabelas
# ============================================================

def postprocess_instrumento(
    work: DataFrame,
    generated: Mapping[str, DataFrame],
) -> DataFrame:
    """
    Recompõe campos derivados da tabela instrumento:

        COD_IF   = COD_TIPO_IF || lpad(NUM_IF, 6, '0')
        COD_ISIN = 'BR' || lpad(NUM_IF, 10, '0')

    Requer que tipo_if já tenha sido gerada.
    """
    if "tipo_if" not in generated:
        raise ValueError(
            "postprocess_instrumento requer tabela `tipo_if` já gerada."
        )

    tipo_if = generated["tipo_if"]

    required_tipo_cols = {"NUM_TIPO_IF", "COD_TIPO_IF"}
    missing_tipo_cols = required_tipo_cols - set(tipo_if.columns)

    if missing_tipo_cols:
        raise ValueError(
            f"Tabela tipo_if não contém colunas necessárias: {missing_tipo_cols}"
        )

    if "NUM_TIPO_IF" not in work.columns:
        raise ValueError("Tabela instrumento não contém NUM_TIPO_IF.")

    tipo = tipo_if.select("NUM_TIPO_IF", "COD_TIPO_IF").dropDuplicates(["NUM_TIPO_IF"])

    if "COD_TIPO_IF" in work.columns:
        work = work.drop("COD_TIPO_IF")

    work = work.join(
        F.broadcast(tipo),
        on="NUM_TIPO_IF",
        how="left",
    )

    if "COD_IF" in work.columns:
        work = work.withColumn(
            "COD_IF",
            F.concat(
                F.coalesce(F.col("COD_TIPO_IF"), F.lit("UNK")),
                F.lpad(F.col("NUM_IF").cast("string"), 6, "0"),
            ),
        )

    if "COD_ISIN" in work.columns:
        work = work.withColumn(
            "COD_ISIN",
            F.concat(
                F.lit("BR"),
                F.lpad(F.col("NUM_IF").cast("string"), 10, "0"),
            ),
        )

    return work.drop("COD_TIPO_IF")


# ============================================================
# 10. Specs da sua base exemplo
# ============================================================

def build_specs_for_base() -> Dict[str, TableSpec]:
    """
    Specs para:

        tipo_if
        instrumento
        operacao
    """
    return {
        "tipo_if": TableSpec(
            name="tipo_if",
            pk_cols=("NUM_TIPO_IF",),
            static=True,
        ),

        "instrumento": TableSpec(
            name="instrumento",
            pk_cols=("NUM_IF",),
            foreign_keys=(
                ForeignKeySpec(
                    columns=("NUM_TIPO_IF",),
                    parent_table="tipo_if",
                    parent_columns=("NUM_TIPO_IF",),
                ),
            ),
            postprocess=postprocess_instrumento,
        ),

        "operacao": TableSpec(
            name="operacao",
            pk_cols=("NUM_ID_OPERACAO",),
            foreign_keys=(
                ForeignKeySpec(
                    columns=("NUM_IF", "COD_IF"),
                    parent_table="instrumento",
                    parent_columns=("NUM_IF", "COD_IF"),
                ),
            ),
        ),
    }


# ============================================================
# 11. Runner para ler CSVs e executar
# ============================================================

def read_csv_table(
    spark: SparkSession,
    path: str,
    *,
    sep: str = ",",
    infer_schema: bool = True,
) -> DataFrame:
    return (
        spark.read
        .option("header", True)
        .option("inferSchema", infer_schema)
        .option("sep", sep)
        .csv(path)
    )


def run_example(
    spark: SparkSession,
    base_path: str,
    *,
    n_rows_by_table: Optional[Mapping[str, int]] = None,
    seed: int = 42,
    validate_mode: ValidateMode = "full",
    save_path: Optional[str] = None,
    save_format: Literal["csv", "parquet"] = "csv",
) -> Dict[str, DataFrame]:
    """
    Lê os 3 CSVs, sintetiza, valida e opcionalmente salva.
    """
    tipo_if = read_csv_table(spark, f"{base_path}/tipo_if.csv")
    instrumento = read_csv_table(spark, f"{base_path}/instrumento.csv")
    operacao = read_csv_table(spark, f"{base_path}/operacao.csv")

    tables = {
        "tipo_if": tipo_if,
        "instrumento": instrumento,
        "operacao": operacao,
    }

    specs = build_specs_for_base()

    if n_rows_by_table is None:
        n_rows_by_table = {
            "tipo_if": tipo_if.count(),
            "instrumento": instrumento.count(),
            "operacao": operacao.count(),
        }

    synthetic = synthesize_multitable_spark(
        tables=tables,
        specs=specs,
        n_rows_by_table=n_rows_by_table,
        seed=seed,
        append_after_max_pk=True,
        validate_mode=validate_mode,
        nullable_fk_policy="allow_any_null",
        broadcast_fk_counts=False,
        storage_level=StorageLevel.MEMORY_AND_DISK,
        verbose=True,
    )

    print("\n>>> Diagnóstico PRIMARY KEYS")
    validate_primary_keys(synthetic, specs).show(truncate=False)

    print("\n>>> Diagnóstico FOREIGN KEYS")
    validate_foreign_keys(
        synthetic,
        specs,
        nullable_fk_policy="allow_any_null",
    ).show(truncate=False)

    print("\n>>> Amostras")
    for name, df in synthetic.items():
        print(f"\n--- {name} ---")
        df.show(5, truncate=False)

    if save_path:
        for name, df in synthetic.items():
            writer = df.coalesce(1).write.mode("overwrite")

            if save_format == "csv":
                writer.option("header", True).csv(f"{save_path}/{name}")
            elif save_format == "parquet":
                writer.parquet(f"{save_path}/{name}")
            else:
                raise ValueError(f"Formato inválido: {save_format}")

        print(f"\nDados sintéticos salvos em: {save_path}")

    return synthetic


# ============================================================


In [26]:
tables = {
    "tipo_if": tipo_if,
    "instrumento": instrumento,
    "operacao": operacao,
}

specs = build_specs_for_base()

#Para testar mais leve, use:
n_rows_by_table = {
    "tipo_if": qtd_tipo_if,
    "instrumento": min(qtd_instrumento, 1000),
    "operacao": min(qtd_operacao, 5000),
}

NameError: name 'qtd_tipo_if' is not defined

In [34]:
# ============================================================
# 5. MONTAR DICT DAS TABELAS
# ============================================================

tables = {
    "tipo_if": tipo_if,
    "instrumento": instrumento,
    "operacao": operacao,
}

specs = build_specs_for_base()


# ============================================================
# 6. DEFINIR QUANTIDADE DE LINHAS SINTÉTICAS
# ============================================================

qtd_tipo_if = tipo_if.count() 
qtd_instrumento = instrumento.count() *10
qtd_operacao = operacao.count() *10

n_rows_by_table = {
    "tipo_if": qtd_tipo_if,
    "instrumento": qtd_instrumento,
    "operacao": qtd_operacao,
}

# Para testar mais leve, use:
# n_rows_by_table = {
#     "tipo_if": qtd_tipo_if,
#     "instrumento": min(qtd_instrumento, 1000),
#     "operacao": min(qtd_operacao, 5000),
# }


# ============================================================
# 7. EXECUTAR SINTETIZAÇÃO
# ============================================================

synthetic = synthesize_multitable_spark(
    tables=tables,
    specs=specs,
    n_rows_by_table=n_rows_by_table,
    seed=42,
    append_after_max_pk=True,
    validate_mode="full",
    nullable_fk_policy="allow_any_null",
    broadcast_fk_counts=False,
    storage_level=StorageLevel.MEMORY_AND_DISK,
    verbose=True,
)

tipo_if_synth = synthetic["tipo_if"]
instrumento_synth = synthetic["instrumento"]
operacao_synth = synthetic["operacao"]

[tipo_if] STATIC | copiando 5 linhas
[instrumento] PAI | origem=1,000 -> sintético=10,000
[operacao] FILHO | origem=50,000 -> sintético=500,000
Validando PKs e FKs...
Validação concluída: PKs e FKs OK.


In [40]:
tipo_if_synth.show(1)

+-----------+-----------+--------------------+-------------------+-------------------+
|NUM_TIPO_IF|COD_TIPO_IF|         NOM_TIPO_IF|       DAT_INCLUSAO|      DAT_ALTERACAO|
+-----------+-----------+--------------------+-------------------+-------------------+
|          1|        CDB|Certificado de De...|2019-12-01 00:00:00|2019-12-01 00:00:00|
+-----------+-----------+--------------------+-------------------+-------------------+
only showing top 1 row



In [41]:
instrumento_synth.show(1)

+------+---------+------------+-----------+-------------------+-----------------+-------------------+-------------------+
|NUM_IF|   COD_IF|    COD_ISIN|NUM_TIPO_IF|VAL_NOMINAL_EMISSAO|VAL_NOMINAL_ATUAL|       DAT_INCLUSAO|      DAT_ALTERACAO|
+------+---------+------------+-----------+-------------------+-----------------+-------------------+-------------------+
|  1001|LCI001001|BR0000001001|          2|          219560.43|        272158.34|2024-10-12 12:17:27|2024-10-27 03:37:09|
+------+---------+------------+-----------+-------------------+-----------------+-------------------+-------------------+
only showing top 1 row



In [42]:
operacao_synth.show(1)

+---------------+------+---------+-------------------+------------+------------------+--------------+---------------------+-------------------+
|NUM_ID_OPERACAO|NUM_IF|   COD_IF|       DAT_OPERACAO|QTD_OPERACAO|VAL_PRECO_UNITARIO|VAL_FINANCEIRO|COD_SITUACAO_OPERACAO|       DAT_INCLUSAO|
+---------------+------+---------+-------------------+------------+------------------+--------------+---------------------+-------------------+
|          50001|  9903|LCI009903|2025-04-21 02:20:08|         268|            9662.8|     2589630.4|                ATIVA|2025-05-31 12:37:16|
+---------------+------+---------+-------------------+------------+------------------+--------------+---------------------+-------------------+
only showing top 1 row



In [38]:
# ============================================================
# 8. VALIDAR PK E FK
# ============================================================

print("VALIDAÇÃO PRIMARY KEYS")
validate_primary_keys(synthetic, specs).show(truncate=False)

print("VALIDAÇÃO FOREIGN KEYS")
validate_foreign_keys(
    synthetic,
    specs,
    nullable_fk_policy="allow_any_null"
).show(truncate=False)

VALIDAÇÃO PRIMARY KEYS
+-----------+---------------+----------+-----------+------------+-----------------+
|table      |pk_cols        |total_rows|distinct_pk|null_pk_rows|duplicate_pk_rows|
+-----------+---------------+----------+-----------+------------+-----------------+
|tipo_if    |NUM_TIPO_IF    |5         |5          |0           |0                |
|instrumento|NUM_IF         |10000     |10000      |0           |0                |
|operacao   |NUM_ID_OPERACAO|500000    |500000     |0           |0                |
+-----------+---------------+----------+-----------+------------+-----------------+

VALIDAÇÃO FOREIGN KEYS
+-----------+-------------+------------+-------------+-----------------+----------+
|child_table|fk_cols      |parent_table|parent_cols  |distinct_child_fk|invalid_fk|
+-----------+-------------+------------+-------------+-----------------+----------+
|instrumento|NUM_TIPO_IF  |tipo_if     |NUM_TIPO_IF  |5                |0         |
|operacao   |NUM_IF,COD_IF|in

In [39]:
# ============================================================
# 9. VISUALIZAR RESULTADOS
# ============================================================

print("TIPO_IF SINTÉTICO")
tipo_if_synth.show(1, truncate=False)

print("INSTRUMENTO SINTÉTICO")
instrumento_synth.show(1, truncate=False)

print("OPERACAO SINTÉTICA")
operacao_synth.show(1, truncate=False)

TIPO_IF SINTÉTICO
+-----------+-----------+--------------------------------------+-------------------+-------------------+
|NUM_TIPO_IF|COD_TIPO_IF|NOM_TIPO_IF                           |DAT_INCLUSAO       |DAT_ALTERACAO      |
+-----------+-----------+--------------------------------------+-------------------+-------------------+
|1          |CDB        |Certificado de Depósito Bancário      |2019-12-01 00:00:00|2019-12-01 00:00:00|
|2          |LCI        |Letra de Crédito Imobiliário          |2019-12-01 00:00:00|2019-12-01 00:00:00|
|3          |LCA        |Letra de Crédito do Agronegócio       |2019-12-01 00:00:00|2019-12-01 00:00:00|
|4          |LF         |Letra Financeira                      |2019-12-01 00:00:00|2019-12-01 00:00:00|
|5          |DPGE       |Depósito a Prazo com Garantia Especial|2019-12-01 00:00:00|2019-12-01 00:00:00|
+-----------+-----------+--------------------------------------+-------------------+-------------------+

INSTRUMENTO SINTÉTICO
+------+------